# VulnReach DAPT Pipeline — Kaggle (2×T4)

**Hardware:** 2× NVIDIA T4 16 GB (fp16, no bf16)  
**GPU quota:** 30 hrs/week free  
**Expected runtime:** ~10–14 hrs total (DAPT 6 000 steps + SFT 2 000 steps)

### Before running
1. Enable GPU: *Settings → Accelerator → GPU T4 × 2*
2. Enable Internet: *Settings → Internet → On*  
   *(needed for model download + corpus fetch — disable after Phase 1 if quota is low)*
3. Push your repo to GitHub and set `REPO_URL` below, **or** upload the repo as a Kaggle dataset and mount it.

In [ ]:
# ── Config — edit these ───────────────────────────────────────────────────────
REPO_URL   = "https://github.com/YOUR_USERNAME/dapt-code.git"  # your repo
WORK_DIR   = "/kaggle/working/dapt-code"

# Kaggle-tuned training knobs (T4 fp16, reduced steps to fit in one session)
DAPT_STEPS  = 6_000   # full spec = 10 000; reduce if session time is tight
DAPT_BATCH  = 2       # 2 per T4 × 2 GPUs × grad_accum 8 = effective 32
SFT_STEPS   = 2_000
SFT_BATCH   = 2
GRAD_ACCUM  = 8

# Set to True to skip corpus download if data/ is already in /kaggle/input
SKIP_INGEST = False

## 0 — Environment check

In [ ]:
import torch, os

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}  {round(p.total_memory/1e9,1)} GB  CC={p.major}.{p.minor}")

assert n_gpus >= 1, "No GPU found — enable GPU in Settings"
assert not torch.cuda.is_bf16_supported(), "bf16 available — you can use an A100 notebook instead"
print(f"\nfp16 training (T4 mode). {n_gpus} GPU(s) detected.")

## 1 — Install dependencies

In [ ]:
# Kaggle pre-installs torch, so we only need the ML training stack on top.
!pip install -q \
    transformers>=4.40.0 \
    datasets>=2.18.0 \
    trl>=0.8.6 \
    peft>=0.10.0 \
    accelerate>=0.28.0 \
    bitsandbytes>=0.43.0 \
    sentencepiece \
    protobuf \
    gitpython \
    tqdm \
    llama-cpp-python

print("Done.")

## 2 — Clone repo

In [ ]:
import os, subprocess

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    print(f"{WORK_DIR} already exists — pulling latest")
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

## Phase 1 — Corpus ingestion
Downloads NVD, OSV, GHSA, CVEfixes. Takes ~60–90 min (network bound).  
**Skip this cell** if you already have `data/dapt_corpus/` from your local machine (upload as a Kaggle dataset and mount it).

In [ ]:
if not SKIP_INGEST:
    !python scripts/dapt/ingest_corpus.py
else:
    print("Skipping ingest — using mounted dataset")
    # If data is in /kaggle/input/dapt-corpus/, symlink it:
    import os
    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/dapt_corpus"):
        os.symlink("/kaggle/input/dapt-corpus", "data/dapt_corpus")
    print("Symlinked /kaggle/input/dapt-corpus → data/dapt_corpus")

## Phase 1b — Merge + tokenize

In [ ]:
!python scripts/dapt/merge_and_tokenize.py

## Phase 3 — Build SFT dataset

In [ ]:
!python scripts/sft/build_sft_dataset.py

## Phase 2 — DAPT training

Runs on **2×T4 via `torchrun`** for ~2× speed vs single GPU.  
`--fp16` forces fp16 (T4 does not support bf16).  
`--max-steps 6000` fits inside a single Kaggle session (~7–9 hrs on 2×T4).  

> **Resume from checkpoint:** If the session dies, re-run this cell — HuggingFace Trainer auto-resumes from the latest checkpoint in `models/dapt_checkpoints/`.

In [ ]:
import subprocess, os

n_gpus = torch.cuda.device_count()
print(f"Launching DAPT on {n_gpus} GPU(s) ...")

cmd = [
    "torchrun",
    f"--nproc_per_node={n_gpus}",
    "scripts/dapt/train_dapt.py",
    f"--max-steps={DAPT_STEPS}",
    f"--batch-size={DAPT_BATCH}",
    f"--grad-accum={GRAD_ACCUM}",
    "--fp16",
]

result = subprocess.run(cmd)
assert result.returncode == 0, "DAPT training failed"

## Phase 4 — SFT training

Merges the DAPT adapter into base weights, then fine-tunes on the security instruction dataset.

In [ ]:
n_gpus = torch.cuda.device_count()
print(f"Launching SFT on {n_gpus} GPU(s) ...")

cmd = [
    "torchrun",
    f"--nproc_per_node={n_gpus}",
    "scripts/sft/train_sft.py",
    f"--max-steps={SFT_STEPS}",
    f"--batch-size={SFT_BATCH}",
    f"--grad-accum={GRAD_ACCUM}",
    "--fp16",
]

result = subprocess.run(cmd)
assert result.returncode == 0, "SFT training failed"

## Phase 5 — Quantize → GGUF

Merges SFT LoRA into full weights and exports Q4_K_M GGUF.  
*(Script not yet written — placeholder cell)*

In [ ]:
import os
if os.path.exists("scripts/export/quantize.py"):
    !python scripts/export/quantize.py
else:
    print("quantize.py not yet written — run Phase 5 manually after implementing it.")

## Save outputs

Kaggle `/kaggle/working/` is wiped after the session.  
Copy the important artefacts to `/kaggle/working/` so they appear in **Output** and can be downloaded.

In [ ]:
import shutil, os

OUT = "/kaggle/working/artefacts"
os.makedirs(OUT, exist_ok=True)

# Always save the adapters — these are small (~150–300 MB each)
for src, name in [
    ("models/dapt_adapter", "dapt_adapter"),
    ("models/sft_final",    "sft_final"),
]:
    dst = os.path.join(OUT, name)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f"Saved {src} → {dst}")

# Save GGUF if it exists
gguf = "models/vulnreach_reranker_q4.gguf"
if os.path.exists(gguf):
    shutil.copy(gguf, os.path.join(OUT, "vulnreach_reranker_q4.gguf"))
    print(f"Saved GGUF ({round(os.path.getsize(gguf)/1e9, 2)} GB)")

print("\nDone. Download from the Output tab on the right.")

## Timing reference (2×T4, fp16)

| Phase | Expected time |
|---|---|
| Phase 1 — corpus ingest | 60–90 min |
| Phase 1b — tokenize | 25–40 min |
| Phase 3 — SFT dataset | 5–10 min |
| Phase 2 — DAPT (6 000 steps) | 5–7 hrs |
| Phase 4 — SFT (2 000 steps) | 1.5–2.5 hrs |
| Phase 5 — quantize | 40–60 min |
| **Total** | **~10–14 hrs** |

Kaggle free sessions cap at ~9–12 hrs. If DAPT runs long, let it checkpoint and resume in a second session with `--from-phase 2` (the Trainer auto-continues from the latest `models/dapt_checkpoints/checkpoint-*/`).

**To resume DAPT from checkpoint**, just re-run the DAPT cell — `torchrun` + HuggingFace Trainer will detect the latest checkpoint automatically.